In [ ]:
!pip install -q transformers peft datasets accelerate evaluate sounddevice scipy librosa
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 81.2 MB/s eta 0:00:00:00:01
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
import datasets
import torch
import random
from transformers import WhisperProcessor, WhisperForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer
from peft import LoraConfig, get_peft_model
from IPython.display import Audio, display

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [ ]:
DATASET_NAME = "MohamedRashad/arabic-english-code-switching"

print("Loading dataset splits...")
dataset_splits = datasets.load_dataset(DATASET_NAME, split={
    'train': 'train[:500]',
    'val': 'train[500:650]',
    'test': 'train[-150:]'
})

train_dataset = dataset_splits['train']
val_dataset = dataset_splits['val']
test_dataset = dataset_splits['test']

print(f"\nTraining set:   {len(train_dataset)} samples")
print(f"Validation set: {len(val_dataset)} samples")
print(f"Final Test set: {len(test_dataset)} samples\n")

train_dataset = train_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
val_dataset = val_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
test_dataset = test_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))

print("\nFiltering out corrupted audio files...")
def is_valid_audio(example):
    try:
        return len(example["audio"]["array"]) > 0
    except Exception:
        return False

train_dataset = train_dataset.filter(is_valid_audio)
val_dataset = val_dataset.filter(is_valid_audio)
test_dataset = test_dataset.filter(is_valid_audio)

print(f"\nSanitized Training set: {len(train_dataset)}")
print(f"Sanitized Validation set: {len(val_dataset)}")
print(f"Sanitized Test set: {len(test_dataset)}\n")

processor = WhisperProcessor.from_pretrained("openai/whisper-large-v3-turbo")

def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=16000
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids
    return batch

print("\nExtracting features for the training set...")
prepared_train_ds = train_dataset.map(prepare_dataset, remove_columns=train_dataset.column_names)

print("Extracting features for the validation set...")
prepared_val_ds = val_dataset.map(prepare_dataset, remove_columns=val_dataset.column_names)

print("Extracting features for the test set...")
prepared_test_ds = test_dataset.map(prepare_dataset, remove_columns=test_dataset.column_names)

print("\nData preparation complete.")

Loading dataset splits...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/2.63G [00:00<?, ?B/s]

data/train-00001-of-00003.parquet:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

data/train-00002-of-00003.parquet:   0%|          | 0.00/2.75G [00:00<?, ?B/s]

...

Extracting features for the validation set...


Map:   0%|          | 0/150 [00:00<?, ? examples/s]

Extracting features for the test set...


Map:   0%|          | 0/150 [00:00<?, ? examples/s]


Data preparation complete.


In [ ]:
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-large-v3-turbo", torch_dtype=torch.float32)

model.requires_grad_(False)

config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)

model = get_peft_model(model, config)
model.print_trainable_parameters()

model.safetensors:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

trainable params: 1,638,400 || all params: 810,516,480 || trainable%: 0.2021


In [ ]:
class DataCollatorSpeechSeq2SeqWithPadding:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, features):
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor)

training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/whisper-lora-checkpoints",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    max_grad_norm=1.0,
    num_train_epochs=10,
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=False,
    logging_steps=10,
    remove_unused_columns=False,
    label_names=["labels"],
)

model.config.use_cache = False
model.gradient_checkpointing_enable()

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=prepared_train_ds,
    eval_dataset=prepared_val_ds,
    data_collator=data_collator,
)

print("Starting LoRA fine-tuning...")
trainer.train()

model.save_pretrained("kaggle/working/whisper-lora-weights")
print("LoRA weights safely saved!")

Starting LoRA fine-tuning...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,14.293797,2.500916
2,8.065829,1.966171
3,6.133613,1.867922
4,5.308175,1.852199
5,4.571397,1.864197
6,4.059670,1.879324
7,3.806930,1.879140
8,3.382555,1.924659
9,3.153219,1.936006
10,3.118954,1.941826


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

LoRA weights safely saved!


In [14]:
NUM_SAMPLES = 25

random_indices = random.sample(range(len(test_dataset)), NUM_SAMPLES)

forced_prompt_ids = processor.get_decoder_prompt_ids(language="arabic", task="transcribe")
model.config.use_cache = True

print(f"Starting evaluation on {NUM_SAMPLES} random samples...\n")

for i, idx in enumerate(random_indices):
    sample = test_dataset[idx]
    
    audio_data = sample["audio"]["array"]
    sampling_rate = sample["audio"]["sampling_rate"]
    ground_truth = sample["sentence"]

    print(f"\n=======================================================")
    print(f" Sample {i + 1} of {NUM_SAMPLES} (Dataset Index: {idx})")
    print(f"=======================================================")

    print("--- 1) Audio File ---")
    display(Audio(data=audio_data, rate=sampling_rate))

    inputs = processor(
        audio_data,
        sampling_rate=sampling_rate,
        return_tensors="pt"
    ).to("cuda", dtype=model.dtype)

    print("Running inference...")

    with torch.no_grad():
        predicted_ids = model.generate(
            inputs.input_features,
            num_beams=5,
            forced_decoder_ids=forced_prompt_ids,
            max_new_tokens=255,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            early_stopping=True
        )

    model_transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

    print("\n--- 2) Model's Transcription ---")
    print(model_transcription)

    print("\n--- 3) Ground-Truth Transcription ---")
    print(ground_truth)
    print("\n")

Starting evaluation on 25 random samples...


 Sample 1 of 25 (Dataset Index: 56)
--- 1) Audio File ---


Both `max_new_tokens` (=255) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running inference...

--- 2) Model's Transcription ---
 حاجة مثبتة علميًا، نعرف إنها موجودة في الـScience، يعني،

--- 3) Ground-Truth Transcription ---
حاجة مُثبتة علميًا.
نعرف إن هي موجود في الـScience يعني.



 Sample 2 of 25 (Dataset Index: 114)
--- 1) Audio File ---


Both `max_new_tokens` (=255) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running inference...

--- 2) Model's Transcription ---
 بس هو يعني النحلة هو فيه 3 انواع نحلة في الخلية

--- 3) Ground-Truth Transcription ---
بس هو يعني النحل هو في تلت انواع نحل في الخلية



 Sample 3 of 25 (Dataset Index: 71)
--- 1) Audio File ---


...

Running inference...

--- 2) Model's Transcription ---
 انتو معاكو like 50, 60, 70% من ال content بتاع التست مبتلس شارك الحاجة بسيطة

--- 3) Ground-Truth Transcription ---
أنتوا معاكوا like fifty، sixty، seventy percent من ال content بتاع ال test، مافضلش غير حاجة بسيطة



 Sample 24 of 25 (Dataset Index: 70)
--- 1) Audio File ---


Both `max_new_tokens` (=255) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running inference...

--- 2) Model's Transcription ---
 تاوضو

--- 3) Ground-Truth Transcription ---
تواضع



 Sample 25 of 25 (Dataset Index: 37)
--- 1) Audio File ---


Both `max_new_tokens` (=255) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running inference...

--- 2) Model's Transcription ---
 فحسس ان هو شخص اداونيه داية فعشان كده أنا عايز.. يعني عشانكده أعايز أقرأه، أنه هو قالي كتاب دا حلو، مش عارف ايه، وقراوس يعني ياتليه هيفيدك

--- 3) Ground-Truth Transcription ---
فحاسس ان هو شخص اداهوني هدية, فعشان كدا انا عايز .. يعني عشان كده عايز اقراه، ان هو قاللي الكتاب دا حلو ومش عارف ايه, إقراه بس يعني هتلاقيه هيفيدك


